# 02. KuaiRand Random-Prediction Feature Construction

This notebook is now an executable wrapper around the same production builder used on Modal:

```text
modal_build_kuairand_random_features.py
```

The old notebook-local feature builder was intentionally removed because it had drifted from the corrected Modal implementation. In particular, the old local code built some histories after filtering to random rows; the corrected implementation builds sessions, event blocks, and lagged histories on **all standard + random test-period rows first**, then filters to random targets.

Use this notebook for three tasks:

1. optionally launch a fresh Modal feature-construction run using the exact same code path as `modal_build_kuairand_random_features.py`,
2. fetch a completed Modal run back to local disk one file at a time,
3. validate the fetched feature tables and manifest.


## Pipeline Implemented by the Modal Builder

The source-of-truth pipeline does the following:

1. Append `log_standard_4_22_to_5_08_27k_part1.csv`, `log_standard_4_22_to_5_08_27k_part2.csv`, and `log_random_4_22_to_5_08_27k.csv`.
2. Deduplicate on `(user_id, video_id, time_ms, tab)` while preserving every random target row. Standard rows that collide with a random key are dropped.
3. Compute labels and outcomes from the merged test-period log.
4. Join user metadata, video metadata, and KuaiRec-compatible category dictionaries.
5. Build event blocks on `(user_id, time_ms)` over all standard + random rows.
6. Build lagged rolling-30 user/category/author histories from strictly prior event blocks/rows.
7. Filter to random rows only after histories are built.
8. Drop unsafe tag features from model inputs because KuaiRand raw `tag` cannot be reliably matched to KuaiRec `video_tag_id/video_tag_name`.
9. Write the full random feature table, the clean ranking input table, audits, and manifest.

The ranking input keeps realized random-row outcomes such as `watch_ratio` and `y_*` for evaluation, but the model-input feature contract is `prediction_feature_cols` in the manifest.


## Output Files

The completed run is stored under:

```text
Modal Volume recsys-kuairand-random-features:/outputs/<RUN_LABEL>/
```

This notebook downloads the same files to:

```text
python_code_KuaiRand/outputs/modal_feature_runs/<RUN_LABEL>/
```

and, if `COPY_TO_CANONICAL = True`, copies them to:

```text
python_code_KuaiRand/outputs/feature_build/
```

Important final files:

| File | Meaning |
|---|---|
| `kuairand_random_prediction_features.parquet` | Full random-row feature table before final duration/watch-ratio quality filtering. |
| `kuairand_random_prediction_ranking_input.parquet` | Clean random-row table for EU/head-score prediction and ranking. |
| `kuairand_random_prediction_feature_manifest.json` | Feature contract, label list, dedup policy, history policy, and output map. |
| `kuairand_random_dropped_bad_duration.parquet` | Rows removed from the ranking input because duration/watch-ratio was invalid. |
| `audit_*.csv` | Construction audits. |


In [ ]:
from pathlib import Path
import importlib
import json
import shutil
import subprocess
import sys
import time

import polars as pl

pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(80)
pl.Config.set_fmt_str_lengths(120)

PROJECT_ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
ANALYSIS_DIR = PROJECT_ROOT / 'python_code_KuaiRand'
CANONICAL_OUTPUT_DIR = ANALYSIS_DIR / 'outputs' / 'feature_build'
RUN_ARCHIVE_ROOT = ANALYSIS_DIR / 'outputs' / 'modal_feature_runs'
CANONICAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)

MODAL_BUILDER_PATH = PROJECT_ROOT / 'modal_build_kuairand_random_features.py'
if not MODAL_BUILDER_PATH.exists():
    raise FileNotFoundError(MODAL_BUILDER_PATH)

# Default to the reviewed completed run. Change RUN_LABEL before fetching or launching a new run.
RUN_LABEL = 'kuairand_random_features_reviewed_20260706'
HISTORY_WINDOW_K = 30
SAMPLE_N = 1000

# Set RUN_MODAL_BUILD=True only when you want to launch a fresh remote build.
RUN_MODAL_BUILD = False
FORCE_UPLOAD = False
WAIT_FOR_COMPLETION = False
WAIT_TIMEOUT_SECONDS = 60 * 60 * 12

# Fetch/copy controls.
FETCH_OUTPUTS = True
COPY_TO_CANONICAL = True
FORCE_DOWNLOAD = True

MODAL_BIN = shutil.which('modal') or '/opt/anaconda3/bin/modal'
if not Path(MODAL_BIN).exists() and shutil.which(MODAL_BIN) is None:
    raise FileNotFoundError(f'Could not find Modal CLI: {MODAL_BIN}')

print('project root:', PROJECT_ROOT)
print('modal builder:', MODAL_BUILDER_PATH)
print('run label:', RUN_LABEL)
print('modal bin:', MODAL_BIN)
print('canonical output dir:', CANONICAL_OUTPUT_DIR)

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
import modal_build_kuairand_random_features as modal_builder
modal_builder = importlib.reload(modal_builder)

required_attrs = [
    'VOLUME_NAME',
    'REMOTE_OUTPUT_DIR',
    'LOCAL_FILES',
    'REMOTE_FILES',
    'volume',
    'build_kuairand_random_features',
]
missing_attrs = [name for name in required_attrs if not hasattr(modal_builder, name)]
if missing_attrs:
    raise AttributeError(f'Modal builder missing expected attributes: {missing_attrs}')

missing_inputs = [f'{name}: {path}' for name, path in modal_builder.LOCAL_FILES.items() if not Path(path).exists()]
if missing_inputs:
    raise FileNotFoundError('Missing required local inputs:\n' + '\n'.join(missing_inputs))

input_table = pl.DataFrame([
    {
        'name': name,
        'local_path': str(path),
        'remote_path': modal_builder.REMOTE_FILES[name],
        'size_gib': Path(path).stat().st_size / 1024**3,
    }
    for name, path in modal_builder.LOCAL_FILES.items()
])

display(input_table)
print('source-of-truth volume:', modal_builder.VOLUME_NAME)
print('source-of-truth remote output dir:', modal_builder.REMOTE_OUTPUT_DIR)

In [ ]:
EXPECTED_OUTPUT_FILES = ['run_log.json', 'kuairand_test_merged_dedup_base.parquet', 'kuairand_test_random_dedup_base.parquet', 'kuairand_video_metadata_selected.parquet', 'kuairand_event_block_history_features.parquet', 'kuairand_category_history_features.parquet', 'kuairand_author_history_features.parquet', 'kuairand_random_prediction_features.parquet', 'kuairand_random_prediction_ranking_input.parquet', 'kuairand_random_dropped_bad_duration.parquet', 'kuairand_random_prediction_features_sample.csv', 'kuairand_random_prediction_feature_manifest.json', 'kuairand_feature_status.csv', 'audit_deduplication.csv', 'audit_sessions_and_blocks.csv', 'audit_event_block_size_distribution.csv', 'audit_feature_missingness.csv', 'audit_tag_category_mapping.csv', 'audit_tag_category_mapping_summary.csv']

RUN_ARCHIVE_DIR = RUN_ARCHIVE_ROOT / RUN_LABEL
RUN_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
REMOTE_RUN_DIR = f"{modal_builder.REMOTE_OUTPUT_DIR}/{RUN_LABEL}"

print('remote run dir:', f'{modal_builder.VOLUME_NAME}:{REMOTE_RUN_DIR}')
print('local archive dir:', RUN_ARCHIVE_DIR)
print('expected output files:', len(EXPECTED_OUTPUT_FILES))

## Optional: Launch a Fresh Modal Build

This cell invokes `modal_build_kuairand_random_features.py` directly. Leave `RUN_MODAL_BUILD = False` in the setup cell if you only want to fetch and validate an existing run. Set it to `True` to recompute features remotely.

In [ ]:
if RUN_MODAL_BUILD:
    print(f'Uploading/reusing inputs in Modal Volume {modal_builder.VOLUME_NAME}')
    try:
        with modal_builder.volume.batch_upload(force=FORCE_UPLOAD) as batch:
            for name, local_path in modal_builder.LOCAL_FILES.items():
                remote_path = modal_builder.REMOTE_FILES[name]
                print(f'  {name}: {local_path} -> {remote_path} ({Path(local_path).stat().st_size / 1024**3:.2f} GiB)')
                batch.put_file(str(local_path), remote_path)
    except FileExistsError:
        print('Some inputs already exist on the Modal Volume; set FORCE_UPLOAD=True to overwrite them.')

    config = {
        'run_label': RUN_LABEL,
        'history_window_k': int(HISTORY_WINDOW_K),
        'sample_n': int(SAMPLE_N),
    }
    call = modal_builder.build_kuairand_random_features.spawn(config)
    call_id = getattr(call, 'object_id', None) or getattr(call, 'function_call_id', None) or 'unknown'
    print('spawned remote KuaiRand feature build:')
    print(json.dumps({**config, 'call_id': call_id}, indent=2))
    print(f'Remote outputs will be in Modal Volume {modal_builder.VOLUME_NAME}:{REMOTE_RUN_DIR}')

    if WAIT_FOR_COMPLETION:
        print('Waiting for Modal call to complete...')
        run_log = call.get(timeout=WAIT_TIMEOUT_SECONDS)
        print(json.dumps(run_log, indent=2))
else:
    print('RUN_MODAL_BUILD is False; not launching a new Modal job.')
    print('Set RUN_MODAL_BUILD=True to recompute features remotely with the source-of-truth builder.')

## Fetch Modal Outputs

The notebook downloads files one by one instead of downloading the whole directory. This avoids stale zero-byte placeholders if a large directory download is interrupted.

In [ ]:
def run_modal_cli(args, check=True):
    cmd = [MODAL_BIN] + list(args)
    print('+', ' '.join(str(x) for x in cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'Modal command failed with exit code {completed.returncode}: {cmd}')
    return completed

if FETCH_OUTPUTS:
    run_modal_cli(['volume', 'ls', modal_builder.VOLUME_NAME, REMOTE_RUN_DIR])
else:
    print('FETCH_OUTPUTS is False; skipped Modal volume listing.')

In [ ]:
def download_one(filename):
    remote_path = f'{REMOTE_RUN_DIR}/{filename}'
    local_path = RUN_ARCHIVE_DIR / filename
    local_path.parent.mkdir(parents=True, exist_ok=True)
    args = ['volume', 'get']
    if FORCE_DOWNLOAD:
        args.append('--force')
    args += [modal_builder.VOLUME_NAME, remote_path, str(local_path)]
    run_modal_cli(args)
    if not local_path.exists():
        raise FileNotFoundError(local_path)
    if local_path.stat().st_size == 0:
        raise ValueError(f'Downloaded zero-byte file: {local_path}')
    return local_path

if FETCH_OUTPUTS:
    downloaded = []
    for filename in EXPECTED_OUTPUT_FILES:
        downloaded.append(download_one(filename))
    print(f'Downloaded {len(downloaded)} files to {RUN_ARCHIVE_DIR}')
else:
    print('FETCH_OUTPUTS is False; skipped downloads.')

In [ ]:
if COPY_TO_CANONICAL:
    copied = []
    for filename in EXPECTED_OUTPUT_FILES:
        src = RUN_ARCHIVE_DIR / filename
        dst = CANONICAL_OUTPUT_DIR / filename
        if not src.exists():
            raise FileNotFoundError(src)
        shutil.copy2(src, dst)
        copied.append(dst)
    print(f'Copied {len(copied)} files to {CANONICAL_OUTPUT_DIR}')
else:
    print('COPY_TO_CANONICAL is False; archive copy only.')

## Validate Local Outputs

This validation reads every parquet footer and row count. If a download is incomplete or corrupt, this cell fails immediately.

In [ ]:
def validate_output_dir(output_dir):
    output_dir = Path(output_dir)
    missing = [name for name in EXPECTED_OUTPUT_FILES if not (output_dir / name).exists()]
    zero = [name for name in EXPECTED_OUTPUT_FILES if (output_dir / name).exists() and (output_dir / name).stat().st_size == 0]
    if missing or zero:
        raise ValueError({'missing': missing, 'zero_byte': zero})

    parquet_rows = []
    for path in sorted(output_dir.glob('*.parquet')):
        lf = pl.scan_parquet(path)
        n_rows = lf.select(pl.len().alias('n')).collect().item()
        parquet_rows.append({'file': path.name, 'rows': int(n_rows), 'size_gib': path.stat().st_size / 1024**3})
    return pl.DataFrame(parquet_rows).sort('file')

archive_parquet_rows = validate_output_dir(RUN_ARCHIVE_DIR)
print('Archive parquet validation:')
display(archive_parquet_rows)

if COPY_TO_CANONICAL:
    canonical_parquet_rows = validate_output_dir(CANONICAL_OUTPUT_DIR)
    print('Canonical parquet validation:')
    display(canonical_parquet_rows)

## Manifest and Policy Checks

The manifest must agree with the corrected Modal policy: random rows are preserved, histories are built from prior all-row event blocks, unsafe tags are dropped, and the ranking input drops invalid duration/watch-ratio rows.

In [ ]:
manifest_path = CANONICAL_OUTPUT_DIR / 'kuairand_random_prediction_feature_manifest.json'
run_log_path = CANONICAL_OUTPUT_DIR / 'run_log.json'
manifest = json.loads(manifest_path.read_text())
run_log = json.loads(run_log_path.read_text())

expected_policy = {
    'run_label': RUN_LABEL,
    'dedup_key': ['user_id', 'video_id', 'time_ms', 'tab'],
    'history_window_k': HISTORY_WINDOW_K,
    'history_scope': 'lagged_rolling_30_from_all_standard_plus_random_event_blocks',
    'dropped_unsafe_features': ['i_video_tag_id', 'i_video_tag_name'],
    'prediction_drop_features': ['hist_author_recency_days'],
}

checks = {
    'run_label': manifest.get('run_label') == expected_policy['run_label'],
    'dedup_key': manifest.get('dedup_key') == expected_policy['dedup_key'],
    'history_window_k': manifest.get('history_window_k') == expected_policy['history_window_k'],
    'history_scope': manifest.get('history_scope') == expected_policy['history_scope'],
    'dropped_unsafe_features': manifest.get('dropped_unsafe_features') == expected_policy['dropped_unsafe_features'],
    'prediction_drop_features': manifest.get('prediction_drop_features') == expected_policy['prediction_drop_features'],
    'random_rows_preserved': bool(run_log.get('dedup', {}).get('random_rows_preserved_by_policy')),
}
failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise ValueError({'failed_manifest_checks': failed, 'checks': checks})

summary = {
    'run_label': manifest.get('run_label'),
    'dedup_policy': manifest.get('dedup_policy'),
    'history_scope': manifest.get('history_scope'),
    'history_window_k': manifest.get('history_window_k'),
    'same_timestamp_policy': manifest.get('same_timestamp_policy'),
    'drop_policy': manifest.get('drop_policy'),
    'model_features': len(manifest.get('model_feature_cols', [])),
    'prediction_features': len(manifest.get('prediction_feature_cols', [])),
    'continuous': len(manifest.get('continuous_cols', [])),
    'binary': len(manifest.get('binary_cols', [])),
    'categorical': len(manifest.get('categorical_cols', [])),
    'random_rows': run_log.get('random_summary', {}).get('n_random_rows'),
}
print(json.dumps(summary, indent=2))
print('All manifest checks passed.')

## Next Step

Use the clean ranking input table for downstream prediction and ranking:

```text
python_code_KuaiRand/outputs/feature_build/kuairand_random_prediction_ranking_input.parquet
```

That table is the right input for applying:

1. the KuaiRec-pretrained behavior-head MLP from notebook 03,
2. the pretrained EU predictor,
3. the ranking/evaluation comparison between the old behavior-score recommender and the new EU recommender.
